# CNN Time Series Model Testing on Google Colab

This notebook sets up the environment and tests your trained **1D CNN** temporal model (`time_series_cnn_model.py`) on a test HDF5 dataset (e.g., sora2_embeddings.h5).

In [ ]:
# Mount Google Drive to access your data files
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# Configure paths – YOUR ACTUAL PATHS
# Path to your test HDF5 file in Google Drive (e.g., sora2_embeddings.h5)
TEST_HDF5_FILE_PATH = "/content/drive/MyDrive/MIT/Lab/sora2_embeddings.h5"

# Path to your trained CNN checkpoint (.pt file)
CHECKPOINT_PATH = "/content/drive/MyDrive/MIT/Lab/best_model.pt"

# Path to time_series_cnn_model.py in Google Drive (self-contained, no time_series_model)
DRIVE_MODEL_PATH = "/content/drive/MyDrive/MIT/Lab/time_series_cnn_model.py"

# Optional: Dataset filter ("sora2", "avdeepfake1m", "shareveo3", or None for all)
DATASET_FILTER = "sora2"

print("="*60)
print("CONFIGURED PATHS (CNN Testing)")
print("="*60)
print(f"Test HDF5: {TEST_HDF5_FILE_PATH}")
print(f"Checkpoint: {CHECKPOINT_PATH}")
print(f"Model file: {DRIVE_MODEL_PATH}")
print(f"Dataset filter: {DATASET_FILTER}")
print("="*60)

In [ ]:
# Copy time_series_cnn_model.py from Google Drive (self-contained, no other deps)
import shutil, os

if os.path.exists(DRIVE_MODEL_PATH):
    os.makedirs("/content/models", exist_ok=True)
    shutil.copy(DRIVE_MODEL_PATH, "/content/models/time_series_cnn_model.py")
    print("✓ Copied time_series_cnn_model.py from Drive to /content/models/")
    MODEL_FILE = "/content/models/time_series_cnn_model.py"
else:
    print(f"⚠️  File not found: {DRIVE_MODEL_PATH}")
    print("   Upload time_series_cnn_model.py to Drive and set DRIVE_MODEL_PATH")
    MODEL_FILE = None

In [ ]:
# Install required packages
!pip install h5py numpy scikit-learn tqdm
!pip install torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu118

In [ ]:
# Verify GPU is available
import torch

print(f"PyTorch: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
else:
    print("⚠️  No GPU – testing will be slower. Runtime > Change runtime type > GPU")

In [ ]:
# Verify test HDF5 file exists and check its structure
import h5py, os

if os.path.exists(TEST_HDF5_FILE_PATH):
    print(f"✓ Test HDF5 found: {TEST_HDF5_FILE_PATH}")
    print(f"  Size: {os.path.getsize(TEST_HDF5_FILE_PATH)/1e9:.2f} GB")
    try:
        with h5py.File(TEST_HDF5_FILE_PATH, 'r') as f:
            def _p(name, obj): print(f"  {name}: {type(obj).__name__}")
            f.visititems(_p)
            if 'videos' in f:
                n = len(f['videos'])
                print(f"\n  Videos: {n}")
                if n > 0:
                    k = list(f['videos'].keys())[0]
                    v = f['videos'][k]
                    if 'dataset' in v.attrs:
                        d = v.attrs['dataset']
                        print(f"  First video dataset: {d.decode() if isinstance(d, bytes) else d}")
                    if 'embeddings' in v:
                        print(f"  Embeddings: {list(v['embeddings'].keys())}")
                        if 'openl3' in v['embeddings']:
                            print(f"    openl3 shape: {v['embeddings']['openl3'].shape}")
                        if 'senet' in v['embeddings']:
                            print(f"    senet shape: {v['embeddings']['senet'].shape}")
    except Exception as e:
        print(f"  ⚠️  Error: {e}")
        import traceback; traceback.print_exc()
else:
    print(f"⚠️  Test HDF5 not found: {TEST_HDF5_FILE_PATH}")

In [ ]:
# Verify checkpoint file exists
import os

if os.path.exists(CHECKPOINT_PATH):
    print(f"✓ Checkpoint found: {CHECKPOINT_PATH}")
    print(f"  Size: {os.path.getsize(CHECKPOINT_PATH)/1e6:.2f} MB")
    try:
        import torch
        ck = torch.load(CHECKPOINT_PATH, map_location='cpu', weights_only=False)
        print("  Keys:", list(ck.keys()))
        if 'epoch' in ck: print(f"  Epoch: {ck['epoch']}")
        if 'val_auroc' in ck: print(f"  Val AUROC: {ck['val_auroc']:.4f}")
        if 'model_state_dict' in ck:
            n = sum(p.numel() for p in ck['model_state_dict'].values())
            print(f"  Params: {n:,}")
    except Exception as e:
        print(f"  ⚠️  Error: {e}")
else:
    print(f"⚠️  Checkpoint not found: {CHECKPOINT_PATH}")

In [ ]:
# Check label distribution in test dataset (helps diagnose AUROC=0.0)
import h5py, numpy as np, os

print("="*60)
print("LABEL DISTRIBUTION IN TEST DATASET")
print("="*60)

if os.path.exists(TEST_HDF5_FILE_PATH):
    try:
        with h5py.File(TEST_HDF5_FILE_PATH, 'r') as f:
            if 'videos' not in f:
                print("⚠️  No 'videos' group")
            else:
                keys = list(f['videos'].keys())
                print(f"Total videos: {len(keys)}")
                all_labels = []
                datasets = set()
                for k in keys[:50]:
                    v = f['videos'][k]
                    if 'dataset' in v.attrs:
                        d = v.attrs['dataset']
                        d = d.decode() if isinstance(d, bytes) else d
                        datasets.add(d.lower())
                        if DATASET_FILTER and DATASET_FILTER.lower() not in d.lower():
                            continue
                    if 'labels' not in v:
                        continue
                    lb = v['labels']
                    typ = 'audio' if 'audio' in lb else ('video' if 'video' in lb else None)
                    if typ:
                        arr = lb[typ][:]
                        if arr.size:
                            all_labels.extend(arr[0].flatten())
                if all_labels:
                    all_labels = np.array(all_labels)
                    u, c = np.unique(all_labels, return_counts=True)
                    print(f"\nFrom {len(all_labels)} segments, datasets: {list(datasets)}")
                    for val, cnt in zip(u, c):
                        name = "REAL" if val > 0.5 else "FAKE"
                        print(f"  {name} (val={val:.3f}): {cnt:,} ({100*cnt/len(all_labels):.2f}%)")
                    bin_u = np.unique((all_labels > 0.5).astype(int))
                    if len(bin_u) == 1:
                        print("\n  ⚠️  Only ONE class → AUROC will be 0.0 (Accuracy/Loss still valid)")
                    else:
                        print("  ✓ Both classes → AUROC calculable")
                else:
                    print("⚠️  No labels in sampled videos")
    except Exception as e:
        print(f"⚠️  Error: {e}")
        import traceback; traceback.print_exc()
else:
    print(f"⚠️  HDF5 not found: {TEST_HDF5_FILE_PATH}")
print("="*60)

In [ ]:
# Run the CNN test script (subprocess)
import subprocess, sys, os

if MODEL_FILE and os.path.exists(MODEL_FILE):
    if os.path.exists(TEST_HDF5_FILE_PATH) and os.path.exists(CHECKPOINT_PATH):
        print("="*60)
        print("Starting CNN Model Testing")
        print("="*60)
        print(f"Model: {MODEL_FILE}")
        print(f"HDF5: {TEST_HDF5_FILE_PATH}")
        print(f"Checkpoint: {CHECKPOINT_PATH}")
        if DATASET_FILTER:
            print(f"Filter: {DATASET_FILTER}")
        print("="*60 + "\n")

        cmd = [sys.executable, MODEL_FILE, "test", TEST_HDF5_FILE_PATH, CHECKPOINT_PATH]
        if DATASET_FILTER:
            cmd.append(DATASET_FILTER)

        r = subprocess.run(cmd, capture_output=False, text=True)
        print("\n✓ Testing completed" if r.returncode == 0 else f"\n⚠️  Exit code {r.returncode}")
    else:
        missing = [n for n, p in [("Test HDF5", TEST_HDF5_FILE_PATH), ("Checkpoint", CHECKPOINT_PATH)] if not os.path.exists(p)]
        print("⚠️  Missing:", missing)
else:
    print("⚠️  Model file not found. Run the copy cell above.")

In [ ]:
# Alternative: Test via Python import (interactive, inspect metrics)
import sys, os

sys.path.insert(0, '/content/models')

if MODEL_FILE and os.path.exists(MODEL_FILE):
    if os.path.exists(TEST_HDF5_FILE_PATH) and os.path.exists(CHECKPOINT_PATH):
        print("="*60)
        print("CNN Model Testing (Interactive)")
        print("="*60)

        from time_series_cnn_model import test_main

        test_metrics = test_main(
            hdf5_path=TEST_HDF5_FILE_PATH,
            checkpoint_path=CHECKPOINT_PATH,
            filter_dataset=DATASET_FILTER,
            audio_embedding_type="openl3",
            video_embedding_type="senet",
            use_audio_labels=True,
            batch_size=16,
        )

        print("\n" + "="*60)
        print("FINAL TEST RESULTS")
        print("="*60)
        print(f"Test Loss:    {test_metrics['loss']:.4f}")
        print(f"Test AUROC:   {test_metrics['auroc']:.4f}")
        print(f"Test Accuracy: {test_metrics['accuracy']:.4f}")
        print("="*60)
    else:
        print("⚠️  Missing HDF5 or checkpoint. Check paths above.")
else:
    print("⚠️  Model file not found. Check DRIVE_MODEL_PATH.")

## Notes

1. **GPU**: Use a GPU runtime for faster testing (Runtime > Change runtime type > GPU).
2. **Test HDF5**: Upload your test embeddings (e.g. `sora2_embeddings.h5`) to Drive and set `TEST_HDF5_FILE_PATH`.
3. **Checkpoint**: Use a **CNN** checkpoint (`best_model.pt` from `time_series_cnn_model` training). Transformer checkpoints are not compatible.
4. **Model file**: `time_series_cnn_model.py` is self-contained; no `time_series_model` needed.
5. **Dataset filter**: `"sora2"`, `"avdeepfake1m"`, `"shareveo3"`, or `None` for all.

## Setup Checklist

- [ ] Mount Drive
- [ ] Set `TEST_HDF5_FILE_PATH`, `CHECKPOINT_PATH`, `DRIVE_MODEL_PATH`
- [ ] Set `DATASET_FILTER` (e.g. `"sora2"` or `None`)
- [ ] GPU runtime
- [ ] Run cells in order